## Local Volatility Monte Carlo
This notebook contains a Monte Carlo simulation of a local volatility stock price model. The model is used to generate a grid of option prices that will be used in the *Deep Local Volatility Model* example. Here a defined analytic function is used for the local volatility and the DNN will be tasked with replicating this from the synethic option prices generated here.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd
from py_vollib.black_scholes.implied_volatility import implied_volatility
import QuantLib as ql
import math

In [ ]:
# Parameters
S0 = 1.0
N = 21
M = 11
K = torch.linspace(0.5, 1.5, N)  # strike prices
T = torch.linspace(0.5, 2.0, M)  # maturities
r = 0.00  # risk-free rate
dt = 0.01  # time increment
sdt = np.sqrt(dt)
n_steps = int(T.max() / dt)  # number of time steps
n_simulations = 10000000  # number of simulations

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

### Local Volatility Function

In [ ]:
# Local volatility function sigma(S, t)
def sigma(S, t):
    y = (t + 0.1) * torch.sqrt(S + 0.1)
    return 0.3 + y * torch.exp(-y)

In [ ]:
def sigmanp(S, t):
    y = (t + 0.1) * np.sqrt(S + 0.1)
    return 0.3 + y * np.exp(-y)

In [ ]:
# Define the range for S and t
S_range = np.linspace(0.5, 1.5, 100)
t_range = np.linspace(0.5, 2.0, 100)

# Create a meshgrid for S and t
S, t = np.meshgrid(S_range, t_range)

# Compute the local volatility values
vol_values = sigmanp(S, t)

In [ ]:
# Function to plot and save the surface plot
def plot_and_save(filename, cmap, display=True):
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(S, t, vol_values, cmap=cmap, edgecolor='none')
    ax.set_xlabel('Stock Price S')
    ax.set_ylabel('Time t')
    ax.set_zlabel('Local Volatility σ(S,t)')
    fig.colorbar(surf)
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    if not display:
        plt.close(fig)

In [ ]:
plot_and_save('local_volatility_color.png', 'viridis')

In [ ]:
plot_and_save('local_volatility_grayscale.png', 'gray', False)

In [ ]:
def plot_and_save_hm(filename, cmap, vmin=0.4, vmax=0.7, display=True, n = 10):
    fig, ax = plt.subplots(figsize=(14, 10))
    cax = sns.heatmap(vol_values, ax=ax, cmap=cmap, cbar=True, vmin=vmin, vmax=vmax)
    ax.set_xlabel('Stock Price S')
    ax.set_ylabel('Time t')
    
    Sr = np.round(S_range, 2)
    tr = np.round(t_range, 2)
    
    # Set x-ticks and y-ticks to show only every nth label
    xticks = range(len(Sr))
    yticks = range(len(tr))
    
    # Define which ticks to show labels for
    xticklabels = [label if i % n == 0 else '' for i, label in enumerate(Sr)]
    yticklabels = [label if i % n == 0 else '' for i, label in enumerate(tr)]

    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels)
    ax.set_yticks(yticks)
    ax.set_yticklabels(yticklabels)
    
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    if not display:
        plt.close(fig)

In [ ]:
plot_and_save_hm('local_volatility_colorhm.png', 'viridis')

In [ ]:
plot_and_save_hm('local_volatility_grayhm.png', 'gray', 0.4, 0.7, False)

### Monte Carlo Simulation

In [ ]:
S = torch.full((n_simulations, n_steps), S0, device=device)

In [ ]:
for t in range(1, n_steps):
    z = torch.randn(n_simulations, device=device)
    mean_z = torch.mean(z)
    stdev_z = torch.std(z)
    z = (z - mean_z) / stdev_z
    lv = sigma(S[:, t - 1], t * dt)
    lv2 = lv**2
    S[:, t] = S[:, t - 1] * torch.exp((r - 0.5 * lv2) * dt 
                                      + lv * sdt * z)

In [ ]:
S_numpy = S.detach().cpu().numpy()

In [ ]:
len(S_numpy[0,:])

In [ ]:
# Plot a subset of the paths
plt.figure(figsize=(10, 6))
for i in range(min(100, n_simulations)):
    plt.plot(np.linspace(0, 1.5, n_steps), S_numpy[i, :], lw=1)

plt.xlabel('Time (t)')
plt.ylabel('Stock Price (S)')
plt.grid(True)
plt.savefig('MCPaths.png', dpi=300, bbox_inches='tight')

In [ ]:
# Function to price options for a grid of strikes and maturities
def price_options(S, K, T, r, option_type='call'):
    prices = torch.zeros((len(K), len(T)), device=device)
    zeros = torch.zeros((n_simulations), device=device)
    for i, k in enumerate(K):
        for j, t in enumerate(T):
            payoff = torch.zeros(n_simulations, device=device)
            if option_type == 'call':
                payoff = torch.maximum(S[:, t] - k, zeros)
            elif option_type == 'put':
                payoff = torch.maximum(k - S[:, t], zeros)
            prices[i, j] = torch.mean(payoff) * torch.exp(-r * t)
    return prices.detach().cpu().numpy()

In [ ]:
K

In [ ]:
T_steps = T / dt


In [ ]:
T_steps = T_steps.type(torch.int64)
T_steps = T_steps - 1
T_steps

In [ ]:
option_prices = price_options(S, K, T_steps, r, option_type='put')

In [ ]:
prices_flat = option_prices.flatten()
KT_pairs = [(k.item(), t.item())  for k in K for t in T]
K_flat, T_flat = zip(*KT_pairs)
df = pd.DataFrame({
    'Price': prices_flat,
    'K': K_flat,
    'T': T_flat
})

In [ ]:
df

In [ ]:
df['IntrinsicValue'] = df['K'].apply(lambda K: max(K - S0, 0))
df['Price'] = df.apply(lambda row: max(row['Price'], row['IntrinsicValue']), axis=1)
df.drop('IntrinsicValue', axis=1, inplace=True)

In [ ]:
df['ImpliedVol'] = df.apply(lambda row: implied_volatility(row['Price'], S0, row['K'], row['T'], r, 'p'), axis=1)

In [ ]:
def black_scholes_put_derivative(S, K, T, r, sigma):
    strike_put = ql.PlainVanillaPayoff(ql.Option.Put, K)
    stddev = sigma * math.sqrt(T)
    black_put = ql.BlackCalculator(strike_put, 
                                    S, 
                                    stddev, 
                                    1.0)
    black_put_theta = black_put.theta(S, T)
    black_put_strikessens = black_put.strikeSensitivity()

    return black_put_strikessens, black_put_theta

In [ ]:
df[['dC_dK', 'dC_dT']] = df.apply(lambda row: black_scholes_put_derivative(S0, row['K'], row['T'], r, 
                                                                     row['ImpliedVol']), axis=1, 
                                  result_type='expand')

In [ ]:
df.head()

In [ ]:
df.to_csv('LVOptionPrices.csv', index=False)

In [ ]:
def plotimpvol(filename, cmap, display=True):
    # Pivot the DataFrame to create a grid of implied volatilities
    vol_surface = df.pivot(index='K', columns='T', values='ImpliedVol')

    # Creating the 3D plot
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111, projection='3d')

    # Preparing the data for plotting
    X, Y = np.meshgrid(vol_surface.columns, vol_surface.index)
    Z = vol_surface.values

    # Plotting the surface
    surf = ax.plot_surface(X, Y, Z, cmap=cmap, edgecolor='none', antialiased=False)

    # Adding labels and title
    ax.set_xlabel('Maturity T')
    ax.set_ylabel('Strike K')
    ax.set_zlabel('Implied Volatility')

    # Adding a color bar which maps values to colors
    fig.colorbar(surf, shrink=0.5, aspect=5)
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    if not display:
        plt.close(fig)

In [ ]:
def plotprices(filename, cmap, display=True):
    # Pivot the DataFrame to create a grid of implied volatilities
    vol_surface = df.pivot(index='K', columns='T', values='Price')

    # Creating the 3D plot
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111, projection='3d')

    # Preparing the data for plotting
    X, Y = np.meshgrid(vol_surface.columns, vol_surface.index)
    Z = vol_surface.values

    # Plotting the surface
    surf = ax.plot_surface(X, Y, Z, cmap=cmap, edgecolor='none', antialiased=False)

    # Adding labels and title
    ax.set_xlabel('Maturity T')
    ax.set_ylabel('Strike K')
    ax.set_zlabel('Price')
    ax.set_xlim(2, 0.5)
    ax.view_init(azim=30)

    # Adding a color bar which maps values to colors
    fig.colorbar(surf, shrink=0.5, aspect=5)
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    if not display:
        plt.close(fig)

In [ ]:
plotimpvol('lvimpvol.png', 'viridis')

In [ ]:
plotimpvol('lvimpvolgrey.png', 'gray', False)

In [ ]:
plotprices('oppriceslv.png', 'viridis')

In [ ]:
plotprices('oppriceslvgray.png', 'gray', False)